In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#!pip install --upgrade transformers

In [ ]:

import pandas as pd
import polars as pl

import numpy as np
import torch
import re
import unicodedata
import string

In [ ]:
print(torch.cuda.is_available())

### データクリーニング（URLや絵文字、#"!などの記号を削除）関数

In [ ]:
#まずは略語や、スラングを展開する。pythonのcontractionsライブラリが便利
#しかしすべての略語やスラングを展開するわけではないため注意
#train_textsには様々な文字が含まれている。それにはURLや絵文字#!$などが含まれている。
#しかし単語ではないため文章の構成要素ではない、つまりこれらはいらない文字であるため消したい
#kaggleのコンペのコードを見るとre.subによる置き換えが用いられているがコードの実行に非常に時間がかかる
#そこでpythonで辞書を作りこれを適用してやる、実はこれだけで高速化が実現できる。
replacement_dict = {
    #step1 文字化けや特殊文字の修正
    # Special characters
    r"\x89Û_": "", 
    r"\x89ÛÒ": "", 
    r"\x89ÛÓ": "", 
    r"\x89ÛÏWhen": "When", 
    r"\x89ÛÏ": "", 
    r"China\x89Ûªs": "China's", 
    r"let\x89Ûªs": "let's", 
    r"\x89Û÷": "", 
    r"\x89Ûª": "", 
    r"\x89Û\x9d": "", 
    r"å_": "", 
    r"\x89Û¢": "", 
    r"\x89Û¢åÊ": "", 
    r"fromåÊwounds": "from wounds", 
    r"åÊ": "", 
    r"åÈ": "", 
    r"JapÌ_n": "Japan", 
    r"Ì©": "e", 
    r"å¨": "", 
    r"SuruÌ¤": "Suruc", 
    r"åÇ": "", 
    r"å£3million": "3 million", 
    r"åÀ": "", 
    

    #2 短縮形の展開
    r"he's": "he is", 
    r"there's": "there is", 
    r"We're": "We are", 
    r"That's": "That is", 
    r"won't": "will not", 
    r"they're": "they are", 
    r"Can't": "Cannot", 
    r"wasn't": "was not", 
    r"don\x89Ûªt": "do not", 
    r"aren't": "are not", 
    r"isn't": "is not", 
    r"What's": "What is", 
    r"haven't": "have not", 
    r"hasn't": "has not", 
    r"There's": "There is", 
    r"He's": "He is", 
    r"It's": "It is", 
    r"You're": "You are", 
    r"I'M": "I am", 
    r"shouldn't": "should not", 
    r"wouldn't": "would not", 
    r"i'm": "I am", 
    r"I\x89Ûªm": "I am", 
    r"I'm": "I am", 
    r"Isn't": "is not", 
    r"Here's": "Here is", 
    r"you've": "you have", 
    r"you\x89Ûªve": "you have", 
    r"we're": "we are", 
    r"what's": "what is", 
    r"couldn't": "could not", 
    r"we've": "we have", 
    r"it\x89Ûªs": "it is", 
    r"doesn\x89Ûªt": "does not", 
    r"It\x89Ûªs": "It is", 
    r"Here\x89Ûªs": "Here is", 
    r"who's": "who is", 
    r"I\x89Ûªve": "I have", 
    r"y'all": "you all", 
    r"can\x89Ûªt": "cannot", 
    r"would've": "would have", 
    r"it'll": "it will", 
    r"we'll": "we will", 
    r"wouldn\x89Ûªt": "would not", 
    r"We've": "We have", 
    r"he'll": "he will", 
    r"Y'all": "You all", 
    r"Weren't": "Were not", 
    r"Didn't": "Did not", 
    r"they'll": "they will", 
    r"they'd": "they would", 
    r"DON'T": "DO NOT", 
    r"That\x89Ûªs": "That is", 
    r"they've": "they have", 
    r"i'd": "I would", 
    r"should've": "should have", 
    r"You\x89Ûªre": "You are", 
    r"where's": "where is", 
    r"Don\x89Ûªt": "Do not", 
    r"we'd": "we would", 
    r"i'll": "I will", 
    r"weren't": "were not", 
    r"They're": "They are", 
    r"Can\x89Ûªt": "Cannot", 
    r"you\x89Ûªll": "you will", 
    r"I\x89Ûªd": "I would", 
    r"let's": "let us", 
    r"it's": "it is", 
    r"can't": "cannot", 
    r"don't": "do not", 
    r"you're": "you are", 
    r"i've": "I have", 
    r"that's": "that is", 
    r"i'll": "I will", 
    r"doesn't": "does not", 
    r"i'd": "I would", 
    r"didn't": "did not", 
    r"ain't": "am not", 
    r"you'll": "you will", 
    r"I've": "I have", 
    r"Don't": "do not", 
    r"I'll": "I will", 
    r"I'd": "I would", 
    r"Let's": "Let us", 
    r"you'd": "You would", 
    r"It's": "It is", 
    r"Ain't": "am not", 
    r"Haven't": "Have not", 
    r"Could've": "Could have", 
    r"youve": "you have", 
    r"donå«t": "do not", 

    # --- 5. ハッシュタグ、スラング、タイポなどの手動修正 ---
    # データセット固有の知識を最大限に活用する部分
    # Character entity references
    r"&gt;": ">", 
    r"&lt;": "<", 
    r"&amp;": "&", 
    
    # Typos, slang and informal abbreviations
    r"w/e": "whatever", 
    r"w/": "with", 
    r"USAgov": "USA government", 
    r"recentlu": "recently", 
    r"Ph0tos": "Photos", 
    r"amirite": "am I right", 
    r"exp0sed": "exposed", 
    r"<3": "love", 
    r"amageddon": "armageddon", 
    r"Trfc": "Traffic", 
    r"8/5/2015": "2015-08-05", 
    r"WindStorm": "Wind Storm", 
    r"8/6/2015": "2015-08-06", 
    r"10:38PM": "10:38 PM", 
    r"10:30pm": "10:30 PM", 
    r"16yr": "16 year", 
    r"lmao": "laughing my ass off",    
    r"TRAUMATISED": "traumatized", 
    
    # Hashtags and usernames
    r"IranDeal": "Iran Deal", 
    r"ArianaGrande": "Ariana Grande", 
    r"camilacabello97": "camila cabello",  
    r"RondaRousey": "Ronda Rousey",      
    r"MTVHottest": "MTV Hottest", 
    r"TrapMusic": "Trap Music", 
    r"ProphetMuhammad": "Prophet Muhammad", 
    r"PantherAttack": "Panther Attack", 
    r"StrategicPatience": "Strategic Patience", 
    r"socialnews": "social news", 
    r"NASAHurricane": "NASA Hurricane", 
    r"onlinecommunities": "online communities", 
    r"humanconsumption": "human consumption", 
    r"Typhoon-Devastated": "Typhoon Devastated", 
    r"Meat-Loving": "Meat Loving", 
    r"facialabuse": "facial abuse", 
    r"LakeCounty": "Lake County", 
    r"BeingAuthor": "Being Author", 
    r"withheavenly": "with heavenly", 
    r"thankU": "thank you", 
    r"iTunesMusic": "iTunes Music", 
    r"OffensiveContent": "Offensive Content", 
    r"WorstSummerJob": "Worst Summer Job", 
    r"HarryBeCareful": "Harry Be Careful", 
    r"NASASolarSystem": "NASA Solar System", 
    r"animalrescue": "animal rescue", 
    r"KurtSchlichter": "Kurt Schlichter", 
    r"aRmageddon": "armageddon", 
    r"Throwingknifes": "Throwing knives", 
    r"GodsLove": "God's Love", 
    r"bookboost": "book boost", 
    r"ibooklove": "I book love", 
    r"NestleIndia": "Nestle India", 
    r"realDonaldTrump": "Donald Trump", 
    r"DavidVonderhaar": "David Vonderhaar", 
    r"CecilTheLion": "Cecil The Lion", 
    r"weathernetwork": "weather network", 
    r"withBioterrorism&use": "with Bioterrorism & use", 
    r"Hostage&2": "Hostage & 2", 
    r"GOPDebate": "GOP Debate", 
    r"RickPerry": "Rick Perry", 
    r"frontpage": "front page", 
    r"NewsInTweets": "News In Tweets", 
    r"ViralSpell": "Viral Spell", 
    r"til_now": "until now", 
    r"volcanoinRussia": "volcano in Russia", 
    r"ZippedNews": "Zipped News", 
    r"MicheleBachman": "Michele Bachman", 
    r"53inch": "53 inch", 
    r"KerrickTrial": "Kerrick Trial", 
    r"abstorm": "Alberta Storm", 
    r"Beyhive": "Beyonce hive", 
    r"IDFire": "Idaho Fire", 
    r"DETECTADO": "Detected", 
    r"RockyFire": "Rocky Fire", 
    r"Listen/Buy": "Listen / Buy", 
    r"NickCannon": "Nick Cannon", 
    r"FaroeIslands": "Faroe Islands", 
    r"yycstorm": "Calgary Storm", 
    r"IDPs:": "Internally Displaced People :", 
    r"ArtistsUnited": "Artists United", 
    r"ClaytonBryant": "Clayton Bryant", 
    r"jimmyfallon": "jimmy fallon", 
    r"justinbieber": "justin bieber",   
    r"UTC2015": "UTC 2015", 
    r"Time2015": "Time 2015", 
    r"djicemoon": "dj icemoon", 
    r"LivingSafely": "Living Safely", 
    r"FIFA16": "Fifa 2016", 
    r"thisiswhywecanthavenicethings": "this is why we cannot have nice things", 
    r"bbcnews": "bbc news", 
    r"UndergroundRailraod": "Underground Railraod", 
    r"c4news": "c4 news", 
    r"OBLITERATION": "obliteration", 
    r"MUDSLIDE": "mudslide", 
    r"NoSurrender": "No Surrender", 
    r"NotExplained": "Not Explained", 
    r"greatbritishbakeoff": "great british bake off", 
    r"LondonFire": "London Fire", 
    r"KOTAWeather": "KOTA Weather", 
    r"LuchaUnderground": "Lucha Underground", 
    r"KOIN6News": "KOIN 6 News", 
    r"LiveOnK2": "Live On K2", 
    r"9NewsGoldCoast": "9 News Gold Coast", 
    r"nikeplus": "nike plus", 
    r"david_cameron": "David Cameron", 
    r"peterjukes": "Peter Jukes", 
    r"JamesMelville": "James Melville", 
    r"megynkelly": "Megyn Kelly", 
    r"cnewslive": "C News Live", 
    r"JamaicaObserver": "Jamaica Observer", 
    r"TweetLikeItsSeptember11th2001": "Tweet like it is september 11th 2001", 
    r"cbplawyers": "cbp lawyers", 
    r"fewmoretweets": "few more tweets", 
    r"BlackLivesMatter": "Black Lives Matter", 
    r"cjoyner": "Chris Joyner", 
    r"ENGvAUS": "England vs Australia", 
    r"ScottWalker": "Scott Walker", 
    r"MikeParrActor": "Michael Parr", 
    r"4PlayThursdays": "Foreplay Thursdays", 
    r"TGF2015": "Tontitown Grape Festival", 
    r"realmandyrain": "Mandy Rain", 
    r"GraysonDolan": "Grayson Dolan", 
    r"ApolloBrown": "Apollo Brown", 
    r"saddlebrooke": "Saddlebrooke", 
    r"TontitownGrape": "Tontitown Grape", 
    r"AbbsWinston": "Abbs Winston", 
    r"ShaunKing": "Shaun King", 
    r"MeekMill": "Meek Mill", 
    r"TornadoGiveaway": "Tornado Giveaway", 
    r"GRupdates": "GR updates", 
    r"SouthDowns": "South Downs", 
    r"braininjury": "brain injury", 
    r"auspol": "Australian politics", 
    r"PlannedParenthood": "Planned Parenthood", 
    r"calgaryweather": "Calgary Weather", 
    r"weallheartonedirection": "we all heart one direction", 
    r"edsheeran": "Ed Sheeran", 
    r"TrueHeroes": "True Heroes", 
    r"S3XLEAK": "sex leak", 
    r"ComplexMag": "Complex Magazine", 
    r"TheAdvocateMag": "The Advocate Magazine", 
    r"CityofCalgary": "City of Calgary", 
    r"EbolaOutbreak": "Ebola Outbreak", 
    r"SummerFate": "Summer Fate", 
    r"RAmag": "Royal Academy Magazine", 
    r"offers2go": "offers to go", 
    r"foodscare": "food scare", 
    r"MNPDNashville": "Metropolitan Nashville Police Department", 
    r"TfLBusAlerts": "TfL Bus Alerts", 
    r"GamerGate": "Gamer Gate", 
    r"IHHen": "Humanitarian Relief", 
    r"spinningbot": "spinning bot", 
    r"ModiMinistry": "Modi Ministry", 
    r"TAXIWAYS": "taxi ways", 
    r"Calum5SOS": "Calum Hood", 
    r"po_st": "po.st", 
    r"scoopit": "scoop.it", 
    r"UltimaLucha": "Ultima Lucha", 
    r"JonathanFerrell": "Jonathan Ferrell", 
    r"aria_ahrary": "Aria Ahrary", 
    r"rapidcity": "Rapid City", 
    r"OutBid": "outbid", 
    r"lavenderpoetrycafe": "lavender poetry cafe", 
    r"EudryLantiqua": "Eudry Lantiqua", 
    r"15PM": "15 PM", 
    r"OriginalFunko": "Funko", 
    r"rightwaystan": "Richard Tan", 
    r"CindyNoonan": "Cindy Noonan", 
    r"RT_America": "RT America", 
    r"narendramodi": "Narendra Modi", 
    r"BakeOffFriends": "Bake Off Friends", 
    r"TeamHendrick": "Hendrick Motorsports", 
    r"alexbelloli": "Alex Belloli", 
    r"itsjustinstuart": "Justin Stuart", 
    r"gunsense": "gun sense", 
    r"DebateQuestionsWeWantToHear": "debate questions we want to hear", 
    r"RoyalCarribean": "Royal Carribean", 
    r"samanthaturne19": "Samantha Turner", 
    r"JonVoyage": "Jon Stewart", 
    r"renew911health": "renew 911 health", 
    r"SuryaRay": "Surya Ray", 
    r"pattonoswalt": "Patton Oswalt", 
    r"minhazmerchant": "Minhaz Merchant", 
    r"TLVFaces": "Israel Diaspora Coalition", 
    r"pmarca": "Marc Andreessen", 
    r"pdx911": "Portland Police", 
    r"jamaicaplain": "Jamaica Plain", 
    r"Japton": "Arkansas", 
    r"RouteComplex": "Route Complex", 
    r"INSubcontinent": "Indian Subcontinent", 
    r"NJTurnpike": "New Jersey Turnpike", 
    r"Politifiact": "PolitiFact", 
    r"Hiroshima70": "Hiroshima", 
    r"GMMBC": "Greater Mt Moriah Baptist Church", 
    r"versethe": "verse the", 
    r"TubeStrike": "Tube Strike", 
    r"MissionHills": "Mission Hills", 
    r"ProtectDenaliWolves": "Protect Denali Wolves", 
    r"NANKANA": "Nankana", 
    r"SAHIB": "Sahib", 
    r"PAKPATTAN": "Pakpattan", 
    r"Newz_Sacramento": "News Sacramento", 
    r"gofundme": "go fund me", 
    r"pmharper": "Stephen Harper", 
    r"IvanBerroa": "Ivan Berroa", 
    r"LosDelSonido": "Los Del Sonido", 
    r"bancodeseries": "banco de series", 
    r"timkaine": "Tim Kaine", 
    r"IdentityTheft": "Identity Theft", 
    r"AllLivesMatter": "All Lives Matter", 
    r"mishacollins": "Misha Collins", 
    r"BillNeelyNBC": "Bill Neely", 
    r"BeClearOnCancer": "be clear on cancer", 
    r"Kowing": "Knowing", 
    r"ScreamQueens": "Scream Queens", 
    r"AskCharley": "Ask Charley", 
    r"BlizzHeroes": "Heroes of the Storm", 
    r"BradleyBrad47": "Bradley Brad", 
    r"HannaPH": "Typhoon Hanna", 
    r"meinlcymbals": "MEINL Cymbals", 
    r"Ptbo": "Peterborough", 
    r"cnnbrk": "CNN Breaking News", 
    r"IndianNews": "Indian News", 
    r"savebees": "save bees", 
    r"GreenHarvard": "Green Harvard", 
    r"StandwithPP": "Stand with planned parenthood", 
    r"hermancranston": "Herman Cranston", 
    r"WMUR9": "WMUR-TV", 
    r"RockBottomRadFM": "Rock Bottom Radio", 
    r"ameenshaikh3": "Ameen Shaikh", 
    r"ProSyn": "Project Syndicate", 
    r"Daesh": "ISIS", 
    r"s2g": "swear to god", 
    r"listenlive": "listen live", 
    r"CDCgov": "Centers for Disease Control and Prevention", 
    r"FoxNew": "Fox News", 
    r"CBSBigBrother": "Big Brother", 
    r"JulieDiCaro": "Julie DiCaro", 
    r"theadvocatemag": "The Advocate Magazine", 
    r"RohnertParkDPS": "Rohnert Park Police Department", 
    r"THISIZBWRIGHT": "Bonnie Wright", 
    r"Popularmmos": "Popular MMOs", 
    r"WildHorses": "Wild Horses", 
    r"FantasticFour": "Fantastic Four", 
    r"HORNDALE": "Horndale", 
    r"PINER": "Piner", 
    r"BathAndNorthEastSomerset": "Bath and North East Somerset", 
    r"thatswhatfriendsarefor": "that is what friends are for", 
    r"residualincome": "residual income", 
    r"YahooNewsDigest": "Yahoo News Digest", 
    r"MalaysiaAirlines": "Malaysia Airlines", 
    r"AmazonDeals": "Amazon Deals", 
    r"MissCharleyWebb": "Charley Webb", 
    r"shoalstraffic": "shoals traffic", 
    r"GeorgeFoster72": "George Foster", 
    r"pop2015": "pop 2015", 
    r"_PokemonCards_": "Pokemon Cards", 
    r"DianneG": "Dianne Gallagher", 
    r"KashmirConflict": "Kashmir Conflict", 
    r"BritishBakeOff": "British Bake Off", 
    r"FreeKashmir": "Free Kashmir", 
    r"mattmosley": "Matt Mosley", 
    r"BishopFred": "Bishop Fred", 
    r"EndConflict": "End Conflict", 
    r"EndOccupation": "End Occupation", 
    r"UNHEALED": "unhealed", 
    r"CharlesDagnall": "Charles Dagnall", 
    r"Latestnews": "Latest news", 
    r"KindleCountdown": "Kindle Countdown", 
    r"NoMoreHandouts": "No More Handouts", 
    r"datingtips": "dating tips", 
    r"charlesadler": "Charles Adler", 
    r"twia": "Texas Windstorm Insurance Association", 
    r"txlege": "Texas Legislature", 
    r"WindstormInsurer": "Windstorm Insurer", 
    r"Newss": "News", 
    r"hempoil": "hemp oil", 
    r"CommoditiesAre": "Commodities are", 
    r"tubestrike": "tube strike", 
    r"JoeNBC": "Joe Scarborough", 
    r"LiteraryCakes": "Literary Cakes", 
    r"TI5": "The International 5", 
    r"thehill": "the hill", 
    r"3others": "3 others", 
    r"stighefootball": "Sam Tighe", 
    r"whatstheimportantvideo": "what is the important video", 
    r"ClaudioMeloni": "Claudio Meloni", 
    r"DukeSkywalker": "Duke Skywalker", 
    r"carsonmwr": "Fort Carson", 
    r"offdishduty": "off dish duty", 
    r"andword": "and word", 
    r"rhodeisland": "Rhode Island", 
    r"easternoregon": "Eastern Oregon", 
    r"WAwildfire": "Washington Wildfire", 
    r"fingerrockfire": "Finger Rock Fire", 
    r"57am": "57 am", 
    r"fingerrockfire": "Finger Rock Fire", 
    r"JacobHoggard": "Jacob Hoggard", 
    r"newnewnew": "new new new", 
    r"under50": "under 50", 
    r"getitbeforeitsgone": "get it before it is gone", 
    r"freshoutofthebox": "fresh out of the box", 
    r"amwriting": "am writing", 
    r"Bokoharm": "Boko Haram", 
    r"Nowlike": "Now like", 
    r"seasonfrom": "season from", 
    r"epicente": "epicenter", 
    r"epicenterr": "epicenter", 
    r"sicklife": "sick life", 
    r"yycweather": "Calgary Weather", 
    r"calgarysun": "Calgary Sun", 
    r"approachng": "approaching", 
    r"evng": "evening", 
    r"Sumthng": "something", 
    r"EllenPompeo": "Ellen Pompeo", 
    r"shondarhimes": "Shonda Rhimes", 
    r"ABCNetwork": "ABC Network", 
    r"SushmaSwaraj": "Sushma Swaraj", 
    r"pray4japan": "Pray for Japan", 
    r"hope4japan": "Hope for Japan", 
    r"Illusionimagess": "Illusion images", 
    r"SummerUnderTheStars": "Summer Under The Stars", 
    r"ShallWeDance": "Shall We Dance", 
    r"TCMParty": "TCM Party", 
    r"marijuananews": "marijuana news", 
    r"onbeingwithKristaTippett": "on being with Krista Tippett", 
    r"Beingtweets": "Being tweets", 
    r"newauthors": "new authors", 
    r"remedyyyy": "remedy", 
    r"44PM": "44 PM", 
    r"HeadlinesApp": "Headlines App", 
    r"40PM": "40 PM", 
    r"myswc": "Severe Weather Center", 
    r"ithats": "that is", 
    r"icouldsitinthismomentforever": "I could sit in this moment forever", 
    r"FatLoss": "Fat Loss", 
    r"02PM": "02 PM", 
    r"MetroFmTalk": "Metro Fm Talk", 
    r"Bstrd": "bastard", 
    r"bldy": "bloody", 
    r"MetrofmTalk": "Metro Fm Talk", 
    r"terrorismturn": "terrorism turn", 
    r"BBCNewsAsia": "BBC News Asia", 
    r"BehindTheScenes": "Behind The Scenes", 
    r"GeorgeTakei": "George Takei", 
    r"WomensWeeklyMag": "Womens Weekly Magazine", 
    r"SurvivorsGuidetoEarth": "Survivors Guide to Earth", 
    r"incubusband": "incubus band", 
    r"Babypicturethis": "Baby picture this", 
    r"BombEffects": "Bomb Effects", 
    r"win10": "Windows 10", 
    r"idkidk": "I do not know I do not know", 
    r"TheWalkingDead": "The Walking Dead", 
    r"amyschumer": "Amy Schumer", 
    r"crewlist": "crew list", 
    r"Erdogans": "Erdogan", 
    r"BBCLive": "BBC Live", 
    r"TonyAbbottMHR": "Tony Abbott", 
    r"paulmyerscough": "Paul Myerscough", 
    r"georgegallagher": "George Gallagher", 
    r"JimmieJohnson": "Jimmie Johnson", 
    r"pctool": "pc tool", 
    r"DoingHashtagsRight": "Doing Hashtags Right", 
    r"ThrowbackThursday": "Throwback Thursday", 
    r"SnowBackSunday": "Snowback Sunday", 
    r"LakeEffect": "Lake Effect", 
    r"RTphotographyUK": "Richard Thomas Photography UK", 
    r"BigBang_CBS": "Big Bang CBS", 
    r"writerslife": "writers life", 
    r"NaturalBirth": "Natural Birth", 
    r"UnusualWords": "Unusual Words", 
    r"wizkhalifa": "Wiz Khalifa", 
    r"acreativedc": "a creative DC", 
    r"vscodc": "vsco DC", 
    r"VSCOcam": "vsco camera", 
    r"TheBEACHDC": "The beach DC", 
    r"buildingmuseum": "building museum", 
    r"WorldOil": "World Oil", 
    r"redwedding": "red wedding", 
    r"AmazingRaceCanada": "Amazing Race Canada", 
    r"WakeUpAmerica": "Wake Up America", 
    r"\\Allahuakbar\\": "Allahu Akbar", 
    r"bleased": "blessed", 
    r"nigeriantribune": "Nigerian Tribune", 
    r"HIDEO_KOJIMA_EN": "Hideo Kojima", 
    r"FusionFestival": "Fusion Festival", 
    r"50Mixed": "50 Mixed", 
    r"NoAgenda": "No Agenda", 
    r"WhiteGenocide": "White Genocide", 
    r"dirtylying": "dirty lying", 
    r"SyrianRefugees": "Syrian Refugees", 
    r"changetheworld": "change the world", 
    r"Ebolacase": "Ebola case", 
    r"mcgtech": "mcg technologies", 
    r"withweapons": "with weapons", 
    r"advancedwarfare": "advanced warfare", 
    r"letsFootball": "let us Football", 
    r"LateNiteMix": "late night mix", 
    r"PhilCollinsFeed": "Phil Collins", 
    r"RudyHavenstein": "Rudy Havenstein", 
    r"22PM": "22 PM", 
    r"54am": "54 AM", 
    r"38am": "38 AM", 
    r"OldFolkExplainStuff": "Old Folk Explain Stuff", 
    r"BlacklivesMatter": "Black Lives Matter", 
    r"InsaneLimits": "Insane Limits", 
    r"youcantsitwithus": "you cannot sit with us", 
    r"2k15": "2015", 
    r"TheIran": "Iran", 
    r"JimmyFallon": "Jimmy Fallon", 
    r"AlbertBrooks": "Albert Brooks", 
    r"defense_news": "defense news", 
    r"nuclearrcSA": "Nuclear Risk Control Self Assessment", 
    r"Auspol": "Australia Politics", 
    r"NuclearPower": "Nuclear Power", 
    r"WhiteTerrorism": "White Terrorism", 
    r"truthfrequencyradio": "Truth Frequency Radio", 
    r"ErasureIsNotEquality": "Erasure is not equality", 
    r"ProBonoNews": "Pro Bono News", 
    r"JakartaPost": "Jakarta Post", 
    r"toopainful": "too painful", 
    r"melindahaunton": "Melinda Haunton", 
    r"NoNukes": "No Nukes", 
    r"curryspcworld": "Currys PC World", 
    r"ineedcake": "I need cake", 
    r"blackforestgateau": "black forest gateau", 
    r"BBCOne": "BBC One", 
    r"AlexxPage": "Alex Page", 
    r"jonathanserrie": "Jonathan Serrie", 
    r"SocialJerkBlog": "Social Jerk Blog", 
    r"ChelseaVPeretti": "Chelsea Peretti", 
    r"irongiant": "iron giant", 
    r"RonFunches": "Ron Funches", 
    r"TimCook": "Tim Cook", 
    r"sebastianstanisaliveandwell": "Sebastian Stan is alive and well", 
    r"Madsummer": "Mad summer", 
    r"NowYouKnow": "Now you know", 
    r"concertphotography": "concert photography", 
    r"TomLandry": "Tom Landry", 
    r"showgirldayoff": "show girl day off", 
    r"Yougslavia": "Yugoslavia", 
    r"QuantumDataInformatics": "Quantum Data Informatics", 
    r"FromTheDesk": "From The Desk", 
    r"TheaterTrial": "Theater Trial", 
    r"CatoInstitute": "Cato Institute", 
    r"EmekaGift": "Emeka Gift", 
    r"LetsBe_Rational": "Let us be rational", 
    r"Cynicalreality": "Cynical reality", 
    r"FredOlsenCruise": "Fred Olsen Cruise", 
    r"NotSorry": "not sorry", 
    r"UseYourWords": "use your words", 
    r"WordoftheDay": "word of the day", 
    r"Dictionarycom": "Dictionary.com", 
    r"TheBrooklynLife": "The Brooklyn Life", 
    r"jokethey": "joke they", 
    r"nflweek1picks": "NFL week 1 picks", 
    r"uiseful": "useful", 
    r"JusticeDotOrg": "The American Association for Justice", 
    r"autoaccidents": "auto accidents", 
    r"SteveGursten": "Steve Gursten", 
    r"MichiganAutoLaw": "Michigan Auto Law", 
    r"birdgang": "bird gang", 
    r"nflnetwork": "NFL Network", 
    r"NYDNSports": "NY Daily News Sports", 
    r"RVacchianoNYDN": "Ralph Vacchiano NY Daily News", 
    r"EdmontonEsks": "Edmonton Eskimos", 
    r"david_brelsford": "David Brelsford", 
    r"TOI_India": "The Times of India", 
    r"hegot": "he got", 
    r"SkinsOn9": "Skins on 9", 
    r"sothathappened": "so that happened", 
    r"LCOutOfDoors": "LC Out Of Doors", 
    r"NationFirst": "Nation First", 
    r"IndiaToday": "India Today", 
    r"HLPS": "helps", 
    r"HOSTAGESTHROSW": "hostages throw", 
    r"SNCTIONS": "sanctions", 
    r"BidTime": "Bid Time", 
    r"crunchysensible": "crunchy sensible", 
    r"RandomActsOfRomance": "Random acts of romance", 
    r"MomentsAtHill": "Moments at hill", 
    r"eatshit": "eat shit", 
    r"liveleakfun": "live leak fun", 
    r"SahelNews": "Sahel News", 
    r"abc7newsbayarea": "ABC 7 News Bay Area", 
    r"facilitiesmanagement": "facilities management", 
    r"facilitydude": "facility dude", 
    r"CampLogistics": "Camp logistics", 
    r"alaskapublic": "Alaska public", 
    r"MarketResearch": "Market Research", 
    r"AccuracyEsports": "Accuracy Esports", 
    r"TheBodyShopAust": "The Body Shop Australia", 
    r"yychail": "Calgary hail", 
    r"yyctraffic": "Calgary traffic", 
    r"eliotschool": "eliot school", 
    r"TheBrokenCity": "The Broken City", 
    r"OldsFireDept": "Olds Fire Department", 
    r"RiverComplex": "River Complex", 
    r"fieldworksmells": "field work smells", 
    r"IranElection": "Iran Election", 
    r"glowng": "glowing", 
    r"kindlng": "kindling", 
    r"riggd": "rigged", 
    r"slownewsday": "slow news day", 
    r"MyanmarFlood": "Myanmar Flood", 
    r"abc7chicago": "ABC 7 Chicago", 
    r"copolitics": "Colorado Politics", 
    r"AdilGhumro": "Adil Ghumro", 
    r"netbots": "net bots", 
    r"byebyeroad": "bye bye road", 
    r"massiveflooding": "massive flooding", 
    r"EndofUS": "End of United States", 
    r"35PM": "35 PM", 
    r"greektheatrela": "Greek Theatre Los Angeles", 
    r"76mins": "76 minutes", 
    r"publicsafetyfirst": "public safety first", 
    r"livesmatter": "lives matter", 
    r"myhometown": "my hometown", 
    r"tankerfire": "tanker fire", 
    r"MEMORIALDAY": "memorial day", 
    r"MEMORIAL_DAY": "memorial day", 
    r"instaxbooty": "instagram booty", 
    r"Jerusalem_Post": "Jerusalem Post", 
    r"WayneRooney_INA": "Wayne Rooney", 
    r"VirtualReality": "Virtual Reality", 
    r"OculusRift": "Oculus Rift", 
    r"OwenJones84": "Owen Jones", 
    r"jeremycorbyn": "Jeremy Corbyn", 
    r"paulrogers002": "Paul Rogers", 
    r"mortalkombatx": "Mortal Kombat X", 
    r"mortalkombat": "Mortal Kombat", 
    r"FilipeCoelho92": "Filipe Coelho", 
    r"OnlyQuakeNews": "Only Quake News", 
    r"kostumes": "costumes", 
    r"YEEESSSS": "yes", 
    r"ToshikazuKatayama": "Toshikazu Katayama", 
    r"IntlDevelopment": "Intl Development", 
    r"ExtremeWeather": "Extreme Weather", 
    r"WereNotGruberVoters": "We are not gruber voters", 
    r"NewsThousands": "News Thousands", 
    r"EdmundAdamus": "Edmund Adamus", 
    r"EyewitnessWV": "Eye witness WV", 
    r"PhiladelphiaMuseu": "Philadelphia Museum", 
    r"DublinComicCon": "Dublin Comic Con", 
    r"NicholasBrendon": "Nicholas Brendon", 
    r"Alltheway80s": "All the way 80s", 
    r"FromTheField": "From the field", 
    r"NorthIowa": "North Iowa", 
    r"WillowFire": "Willow Fire", 
    r"MadRiverComplex": "Mad River Complex", 
    r"feelingmanly": "feeling manly", 
    r"stillnotoverit": "still not over it", 
    r"FortitudeValley": "Fortitude Valley", 
    r"CoastpowerlineTramTr": "Coast powerline", 
    r"ServicesGold": "Services Gold", 
    r"NewsbrokenEmergency": "News broken emergency", 
    r"Evaucation": "evacuation", 
    r"leaveevacuateexitbe": "leave evacuate exit be", 
    r"P_EOPLE": "PEOPLE", 
    r"Tubestrike": "tube strike", 
    r"CLASS_SICK": "CLASS SICK", 
    r"localplumber": "local plumber", 
    r"awesomejobsiri": "awesome job siri", 
    r"PayForItHow": "Pay for it how", 
    r"ThisIsAfrica": "This is Africa", 
    r"crimeairnetwork": "crime air network", 
    r"KimAcheson": "Kim Acheson", 
    r"cityofcalgary": "City of Calgary", 
    r"prosyndicate": "pro syndicate", 
    r"660NEWS": "660 NEWS", 
    r"BusInsMagazine": "Business Insurance Magazine", 
    r"wfocus": "focus", 
    r"ShastaDam": "Shasta Dam", 
    r"go2MarkFranco": "Mark Franco", 
    r"StephGHinojosa": "Steph Hinojosa", 
    r"Nashgrier": "Nash Grier", 
    r"NashNewVideo": "Nash new video", 
    r"IWouldntGetElectedBecause": "I would not get elected because", 
    r"SHGames": "Sledgehammer Games", 
    r"bedhair": "bed hair", 
    r"JoelHeyman": "Joel Heyman", 
    r"viaYouTube": "via YouTube",

    # --- 7. 天気などの略語を展開 ---
    # Acronyms
    r"MH370": "Malaysia Airlines Flight 370",
    r"mÌ¼sica": "music", 
    r"okwx": "Oklahoma City Weather", 
    r"arwx": "Arkansas Weather", 
    r"gawx": "Georgia Weather", 
    r"scwx": "South Carolina Weather", 
    r"cawx": "California Weather", 
    r"tnwx": "Tennessee Weather", 
    r"azwx": "Arizona Weather", 
    r"alwx": "Alabama Weather", 
    r"wordpressdotcom": "wordpress", 
    r"usNWSgov": "United States National Weather Service", 
    r"Suruc": "Sanliurfa", 

    # Grouping same words without embeddings
    r"Bestnaijamade": "bestnaijamade", 
    r"SOUDELOR": "Soudelor", 
    }


In [ ]:
# キーを長い順にソートする（例: "we're"が"we"より先に置換されるようにするため）
sorted_keys = sorted(replacement_dict.keys(), key=len, reverse=True)

# すべてのキーを'|'で連結し、1つの正規表現パターンを作成
# re.escape()で特殊文字をエスケープするのが重要
pattern = re.compile("|".join(map(re.escape, sorted_keys)))

def full_clean(tweet):
    # 1. 辞書を使って一括置換
    # pattern.sub()の第二引数に関数を渡すことで、マッチした部分に応じた置換が可能
    tweet = pattern.sub(lambda m: replacement_dict[m.group(0)], tweet)

    # 2. URLの除去 (これは個別でOK)
    tweet = re.sub(r"https?:\/\/t.co\/[A-Za-z0-9]+", "", tweet)

    # 3. HTMLエンティティの修正
    tweet = tweet.replace("&gt;", ">").replace("&lt;", "<").replace("&amp;", "&")

    # 4. 句読点の周りにスペースを追加
    punctuations = '@#!?+&*[]-%.:/();$=><|{}^' + "'`"
    for p in punctuations:
        tweet = tweet.replace(p, f' {p} ')
    tweet = tweet.replace('...', ' ... ')
    if '...' not in tweet:
        tweet = tweet.replace('..', ' ... ')

    # 5. 連続する空白を1つにまとめる
    tweet = re.sub(r'\s+', ' ', tweet).strip()

    return tweet


### 実際にデータにクリーニングを適用


In [ ]:
train_df = pl.read_csv("../input/nlp-getting-started/train.csv")


#クリーニング関数の適用
train_df = train_df.with_columns([
    pl.col("text").map_elements(full_clean, return_dtype=pl.String).alias("text_cleaned")
])

In [ ]:
print(train_df.head())
print(train_df.tail())

### EDAを行う。まずは、targetの0と1を比較してデータ内の災害ツイートと被災害ツイートがどれほど含まれているかを調べる

In [ ]:
train_df = train_df.to_pandas()

# ターゲット分布
train_df['target'].value_counts().plot(kind='bar', title='Disaster (1) vs Non-Disaster (0)')

### データは若干被災害データに偏っている程度であり、偏りはあまりないと言っていい
### 単語の頻度を分析してみる

In [ ]:
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# --- データ準備 ---
df_disaster = train_df[train_df['target'] == 1]
df_nondisaster = train_df[train_df['target'] == 0]

disaster_texts = " ".join(text for text in df_disaster.text_cleaned)
nondisaster_texts = " ".join(text for text in df_nondisaster.text_cleaned)

# ストップワードの設定
stopwords = set(STOPWORDS)

# --- 2つのワードクラウドを描画 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# 1. 災害ツイートのワードクラウド
wordcloud1 = WordCloud(stopwords=stopwords, background_color='white', width=800, height=800).generate(disaster_texts)
ax1.imshow(wordcloud1, interpolation='bilinear')
ax1.set_title('Disaster Tweets')
ax1.axis('off')

# 2. 非災害ツイートのワードクラウド
wordcloud2 = WordCloud(stopwords=stopwords, background_color='white', width=800, height=800).generate(nondisaster_texts)
ax2.imshow(wordcloud2, interpolation='bilinear')
ax2.set_title('Non-Disaster Tweets')
ax2.axis('off')

plt.show()


### 次は単語を数えてみる

In [ ]:
from collections import Counter
import seaborn as sns

# ストップワードを設定
stopwords = set(STOPWORDS)

# 無視したい句読点のリストを定義
punctuations_to_ignore = set('@#!?+&*[]-%.:/();$=><|{}^' + "'`" + "...")

# --- データ準備 ---
corpus_disaster = train_df[train_df['target']==1]['text_cleaned']
corpus_nondisaster = train_df[train_df['target']==0]['text_cleaned']

# --- 単語リストを作成する際に、ストップワードと句読点を除外 ---
tokens_disaster = [
    word for word in " ".join(corpus_disaster).split() 
    if word.lower() not in stopwords and word not in punctuations_to_ignore
]
tokens_nondisaster = [
    word for word in " ".join(corpus_nondisaster).split() 
    if word.lower() not in stopwords and word not in punctuations_to_ignore
]

# --- 上位20単語の頻度を計算 ---
top_20_disaster = Counter(tokens_disaster).most_common(20)
top_20_nondisaster = Counter(tokens_nondisaster).most_common(20)

# 計算結果をDataFrameに変換
df_disaster_top = pd.DataFrame(top_20_disaster, columns=['word', 'count'])
df_nondisaster_top = pd.DataFrame(top_20_nondisaster, columns=['word', 'count'])

# --- 2つの棒グラフを描画 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# 1. 災害ツイートの棒グラフ
sns.barplot(x='count', y='word', data=df_disaster_top, ax=ax1, palette='Reds_d')
ax1.set_title('Top 20 words in Disaster Tweets')
ax1.set_xlabel('Count')
ax1.set_ylabel('Word')

# 2. 非災害ツイートの棒グラフ
sns.barplot(x='count', y='word', data=df_nondisaster_top, ax=ax2, palette='Blues_d')
ax2.set_title('Top 20 words in Non-Disaster Tweets')
ax2.set_xlabel('Count')
ax2.set_ylabel('')

plt.tight_layout()
plt.show()

### N-gram分析をやってみる、単語ではなくフレーズを数えてみる。まずはbi-gram 2wardでやってみる。

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Bigramを抽出して上位を返す関数を作成
def get_top_2_bigrams(corpus, n=20):
    """
    テキストコーパスから上位n個のBigramを抽出する
    """
    # Bigramをカウント
    vectorizer = CountVectorizer(ngram_range=(2,2), stop_words='english')
    X = vectorizer.fit_transform(corpus)
    
    # トークンの頻度を集計
    sum_words = X.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    
    # DataFrameに変換して返す
    return pd.DataFrame(words_freq[:n], columns=['Bigram', 'Frequency'])

# --- 1. 各クラスの上位Bigramを計算 ---
# 災害ツイート
top_bigrams_disaster = get_top_2_bigrams(train_df[train_df['target']==1]['text_cleaned'])
# 非災害ツイート
top_bigrams_nondisaster = get_top_2_bigrams(train_df[train_df['target']==0]['text_cleaned'])

print("--- Top 20 Bigrams in Disaster Tweets ---")
print(top_bigrams_disaster)
print("\n--- Top 20 Bigrams in Non-Disaster Tweets ---")
print(top_bigrams_nondisaster)

# --- 2. 結果を棒グラフで可視化 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 12))

# 災害ツイートのBigram
sns.barplot(x='Frequency', y='Bigram', data=top_bigrams_disaster, ax=ax1, palette='Reds_d')
ax1.set_title('Top 20 Bigrams in Disaster Tweets')

# 非災害ツイートのBigram
sns.barplot(x='Frequency', y='Bigram', data=top_bigrams_nondisaster, ax=ax2, palette='Blues_d')
ax2.set_title('Top 20 Bigrams in Non-Disaster Tweets')

plt.tight_layout()
plt.show()

In [ ]:
# Bigramを抽出して上位を返す関数を作成
def get_top_3_bigrams(corpus, n=20):
    """
    テキストコーパスから上位n個のBigramを抽出する
    """
    # Bigramをカウント
    vectorizer = CountVectorizer(ngram_range=(3,3), stop_words='english')
    X = vectorizer.fit_transform(corpus)
    
    # トークンの頻度を集計
    sum_words = X.sum(axis=0) 
    words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    
    # DataFrameに変換して返す
    return pd.DataFrame(words_freq[:n], columns=['Trigram', 'Frequency'])

# --- 1. 各クラスの上位Bigramを計算 ---
# 災害ツイート
top_bigrams_disaster = get_top_3_bigrams(train_df[train_df['target']==1]['text_cleaned'])
# 非災害ツイート
top_bigrams_nondisaster = get_top_3_bigrams(train_df[train_df['target']==0]['text_cleaned'])

print("--- Top 20 Trirams in Disaster Tweets ---")
print(top_bigrams_disaster)
print("\n--- Top 20 Trigrams in Non-Disaster Tweets ---")
print(top_bigrams_nondisaster)

# --- 2. 結果を棒グラフで可視化 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 12))

# 災害ツイートのBigram
sns.barplot(x='Frequency', y='Trigram', data=top_bigrams_disaster, ax=ax1, palette='Reds_d')
ax1.set_title('Top 20 Trigrams in Disaster Tweets')

# 非災害ツイートのBigram
sns.barplot(x='Frequency', y='Trigram', data=top_bigrams_nondisaster, ax=ax2, palette='Blues_d')
ax2.set_title('Top 20 Trigrams in Non-Disaster Tweets')

plt.tight_layout()
plt.show()

### bi-gramsとTri-gramsから単純なキーワードの組み合わせだけでは解けないことが分かった。
### つまり文脈を正しく認識できるようなモデルでなければいけない。
### そこでBERTモデルを用いる。

In [ ]:
from transformers import BertTokenizer

### BERTのtokenizerでテキストをトークン化する

In [ ]:
# BERTのTokenizerをロード（小文字化された英語用）
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# サンプルで確認
sample_text = train_df['text_cleaned'].iloc[0]
encoding = tokenizer.encode_plus(
    sample_text,
    add_special_tokens=True,
    max_length=128,
    return_token_type_ids=False,
    padding='max_length',
    truncation=True,
    return_attention_mask=True,
    return_tensors='pt'  # PyTorch用テンソル
)

print("Input IDs:", encoding['input_ids'])
print("Attention Mask:", encoding['attention_mask'])

### データセットに適用

In [ ]:
# トークナイザのロード
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# データセット全体を一度にトークン化
# .tolist()でテキストの列をPythonのリストに変換して渡すのが効率的
encodings = tokenizer(
    train_df['text_cleaned'].tolist(),
    add_special_tokens=True,
    max_length=128,              # ツイートの長さを考慮した最大長
    padding='max_length',        # 最大長までパディング
    truncation=True,             # 最大長を超える場合は切り捨て
    return_attention_mask=True,
    return_tensors='pt'          # PyTorchテンソルとして返す
)

# トークン化されたデータを変数に格納
input_ids = encodings['input_ids']
attention_mask = encodings['attention_mask']

# ラベルもPyTorchテンソルに変換
labels = torch.tensor(train_df['target'].values)

# これで、input_ids, attention_mask, labels の3つがモデルの入力として準備できました。
print('Input IDs shape:', input_ids.shape)
print('Attention Mask shape:', attention_mask.shape)
print('Labels shape:', labels.shape)

### データセット処理を定義

In [ ]:
from torch.utils.data import Dataset

class DisasterTweetDataset(Dataset):
    def __init__(self, texts, targets, tokenizer, max_len):
        """
        初期化メソッド
        :param texts: ツイートのテキスト (Pandas Series)
        :param targets: ラベル (Pandas Series)
        :param tokenizer: BERTのトークナイザ
        :param max_len: テキストの最大長
        """
        self.texts = texts
        self.targets = targets
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        """
        データセットの全件数を返す
        """
        return len(self.texts)

    def __getitem__(self, item):
        """
        指定されたインデックス(item)のデータを取得し、
        BERTに入力できる形式に変換して返す
        """
        text = str(self.texts.iloc[item])
        target = self.targets.iloc[item]

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(target, dtype=torch.long)
        }

### データを分割

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# データを分割
train_texts, val_texts, train_targets, val_targets = train_test_split(
    train_df['text_cleaned'],
    train_df['target'],
    test_size=0.2,
    random_state=42
)

MAX_LEN = 128

train_dataset = DisasterTweetDataset(train_texts, train_targets, tokenizer, MAX_LEN)
val_dataset = DisasterTweetDataset(val_texts, val_targets, tokenizer, MAX_LEN)

### BERTモデルの読み込みとTrainerの設定

In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score

In [ ]:
def compute_metrics(pred):
    """
    予測結果を受け取り、評価指標を計算する関数
    """
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    
    # F1スコアとAccuracyを計算
    f1 = f1_score(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    
    # 辞書形式で返す
    return {
        'accuracy': acc,
        'f1': f1,
    }

In [ ]:
print(TrainingArguments)

In [ ]:
import inspect
print(inspect.signature(TrainingArguments.__init__))

In [ ]:
%%time

# （モデルの定義やデータセットの準備は完了していると仮定）
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    report_to="none",  # ★この行を追加
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',  # エポックごとに評価
    save_strategy='epoch',
    logging_dir='./logs',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1'  # F1スコアで最適なモデルを判断
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics  # ★この行を追加
)

# 訓練を開始
trainer.train()

In [ ]:
# 検証データセットで評価を実行
final_results = trainer.evaluate()

# 結果を表示
print(final_results)

In [ ]:
test_df = pl.read_csv("../input/nlp-getting-started/test.csv")

# 2. DisasterTweetDatasetクラスをテストデータ用にインスタンス化
# テストデータには正解ラベルがないため、ダミーのラベル（例：すべて0）を作成して渡します。
dummy_targets = pd.Series([0] * len(test_df))

#クリーニング関数の適用
test_df = test_df.with_columns([
    pl.col("text").map_elements(full_clean, return_dtype=pl.String).alias("text_cleaned")
])

test_df = test_df.to_pandas()

In [ ]:
test_dataset = DisasterTweetDataset(
    texts=test_df['text_cleaned'], # クリーニング済みのテキスト
    targets=dummy_targets,           # ダミーのラベル
    tokenizer=tokenizer,             # 学習時と同じトークナイザー
    max_len=128                      # 学習時と同じ最大長
)

In [ ]:
# 3. このtest_datasetをTrainerのpredictメソッドに渡す
raw_predictions = trainer.predict(test_dataset)

In [ ]:
# 1. 予測結果（ロジット）から最も確率の高いクラスを予測ラベルとして選び出す
predicted_labels = np.argmax(raw_predictions.predictions, axis=1)

# 2. sample_submission.csvを読み込む
submission_df = pd.read_csv('/kaggle/input/nlp-getting-started/sample_submission.csv')

# 3. 'target'列を予測したラベルで置き換える
submission_df['target'] = predicted_labels


In [ ]:
# 4. 提出用のCSVファイルとして保存する（index=Falseが重要）
submission_df.to_csv('submission.csv', index=False)

print("submission.csvの作成が完了しました。")
print(submission_df.head())